# Script to update metadata for any dataset

Check metadata for old sequences to see if anything has been added

Databases: GISAID, Andersen, NCBI Virus

In [1]:
# Housekeeping


import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

# Dates
start_date = "11-01-2021"
end_date = "06-13-2025"
date_range = start_date + "--" + end_date
update_date = "06-27-2025"

references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/"
os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
# downloads_gisaid = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder for gisaid
# downloads_gisaid = "C:/Users/maksi/Downloads/"
downloads_gisaid = home + "GISAID/downloads/" + date_range + "_Antarctica_North_America_South_America/" # Cats are North America
downloads_ncbi_virus = home + "NCBI_Virus/downloads/" + date_range + "_Antarctica_North_America_South_America/"

genotype = "D1.1"
genotype_underscored = genotype.replace(".", "_")

# originals = home + "Combinations/GISAID_Andersen_NCBI_Virus/" + date_range + "_Antarctica_North_America_South_America/" # "_cats/" # + "_" + genotype_underscored + "/"
originals = home + "Alignments/" + date_range + "_cats/" + genotype_underscored + "/"

# complete = home + "Combinations/GISAID_Andersen_NCBI_Virus/" + date_range + "_Antarctica_North_America_South_America/updated_" + update_date + "/"
complete = originals

os.chdir(originals)

## Original Files

In [2]:
# Function to prepare dataframes
def fasta_df_og(file_name, states_ref):

    fasta = pd.DataFrame()
    headers = []
    isolate_ids = []
    isolate_names = []
    subtypes = []
    # segments = []
    collection_dates = []
    sequences = []
    host_types = []
    species = []
    identifiers = []
    genotypes = []
    name_states = []
    with open(file_name) as f:
        lines = f.readlines()
        for num, line in enumerate(lines):
            # print(line)
            if line[0] == ">": # If it's a header
                if line[1:].strip() not in headers: # And the previous line is not a header we've seen before
                    header = line[1:].strip() # Remove the ">"
                    # print(header)
                    split_header = header.split("|")
                    if len(header.split("|")) > 6:
                            identifier = header.split("|")[0]
                            identifiers.append(identifier)
                            split_first_header = split_header[1].split("/")
                    else:
                        identifiers.append("unknown")
                        split_first_header = split_header[0].split("/")
                    # print(split_first_header)
                    # print(split_header)
                    headers.append(header) 
                    name_states.append(split_first_header[2].replace("_", " "))
                    isolate_ids.append(split_first_header[3])
                    isolate_names.append(split_header[-6]) # We'll need to extract data from this too
                    # print(split_header[2].split("_")[-1])
                    subtypes.append(split_header[-5])  # Get only H5N1
                    genotypes.append(split_header[-1])
                    # segments.append(split_header[].split("_")[-1])
                    host_types.append(split_header[-2])
                    species.append(split_first_header[1])
                    # if split_header[4] == "2024-01-01":
                    #     collection_dates.append("2024") # No samples were collected 1/1/2024, these are all unknown 
                    # elif split_header[4] == "2025-01-01":
                    #     collection_dates.append("2025")
                    # else: 
                    collection_dates.append(split_header[-3].split("_")[-1])
                    if num < len(lines): # If we're not at the last line
                        # for i, l in enumerate(lines[num + 1:]):
                        i = num
                        sequence = ""
                        # print(lines[i])
                        # print(lines[i + 1])
                        while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                            sequence = sequence + lines[i + 1].strip()
                            i += 1
                        sequences.append(sequence) # Add next line to sequences
        f.close()

    # Create columns for data frame 
    fasta["Header"] = headers
    fasta["Isolate_Id"] = isolate_ids
    fasta["Isolate_Name"] = isolate_names
    fasta["Subtype"] = subtypes
    fasta["name_state"] = name_states
    # fasta["Segment"] = segments
    # Geo_Location is more complicated
    fasta["Geo_Location"] = fasta["name_state"].apply(lambda x: 
                                                        states_ref.loc[states_ref["Abbreviation"].str.contains('|'.join(x.replace(', ', ' ').split(' ')), regex=True), 'Country'].iloc[0] 
                                                        + "-" + 
                                                        states_ref.loc[states_ref['Abbreviation'].str.contains('|'.join(x.replace(', ', ' ').split(' ')), regex=True), 'Abbreviation'].iloc[0] 
                                                        if states_ref["Abbreviation"].str.contains("|".join((x.replace(", ", " ").split(" "))), regex=True).any()
                                                        else states_ref.loc[states_ref['State'].str.contains('|'.join(x.replace(', ', ' ').split(' ')), regex=True), 'Country'].iloc[0]
                                                        + "-" + 
                                                        states_ref.loc[states_ref['State'].str.contains('|'.join(x.replace(', ', ' ').split(' ')), regex=True), 'Abbreviation'].iloc[0] 
                                                        if states_ref["State"].str.contains("|".join((x.replace(", ", " ").split(" "))), regex=True).any() 
                                                        else x)
    fasta["Date Collected"] = collection_dates
    fasta["Species"] = species
    fasta["Host_Type"] = host_types
    fasta["Genotype"] = genotypes
    fasta["Sequence"] = sequences
    if len(identifiers) == len(fasta):
        fasta["Identifier"] = identifiers
    
    return fasta

original_fasta_dfs = {}

for dirpath, dirs, files in os.walk(originals):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
            fasta_file = fasta_df_og(file_name, states_ref)
            original_fasta_dfs[file_name] = fasta_file
            # print(fasta_file)
            # break 
    break 

In [3]:
print(list(original_fasta_dfs.keys())[0])
print(original_fasta_dfs[list(original_fasta_dfs.keys())[0]])

C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Alignments/11-01-2021--06-13-2025_cats/D1_1/D1_1_HA_11-01-2021--06-13-2025_aln.fasta
                                                Header  \
0    SRR33124713|A/CATTLE/USA/25-006783-001/2025|H5...   
1    SRR33124725|A/QUAIL/USA/25-006346-009/2025|H5N...   
2    SRR33124734|A/DUCK/USA/25-007078-002/2025|H5N1...   
3    SRR33124735|A/DUCK/USA/25-007078-001/2025|H5N1...   
4    SRR33124737|A/DUCK/USA/25-006674-002/2025|H5N1...   
..                                                 ...   
764  SRR33029749|A/gallus gallus/NY/25-010293-002-o...   
765  SRR33029739|A/gallus gallus/PA/25-010255-001-o...   
766  SRR33029738|A/gallus gallus/PA/25-010255-002-o...   
767  SRR33029737|A/gallus gallus/PA/25-010255-003-o...   
768  SRR33029746|A/anatidae/CA/24-035866-001-origin...   

                 Isolate_Id                                    Isolate_Name  \
0             25-006783-001                 A/CATTLE/USA/25-006783-001/2025   
1     

## GISAID

In [4]:
# # Get data from GISAID

# username = input("Username: ")
# password = input("Password: ")
# browser = input("Browser: ")
# sleep_time = input("Seconds to sleep in between clicks: ")
# continent = input("Continent(s) separated by commas: ")
# start_date = dateutil.parser.parse(start_date).strftime("%Y-%m-%d") # Make sure date is in correct format
# end_date = dateutil.parser.parse(end_date).strftime("%Y-%m-%d")

# open_gisaid(username, password, browser, sleep_time, continent, start_date, end_date)

In [5]:
# Get downloaded GISAID data

all_metadata_files = []
all_fasta_files = []

# Grab files
for dirpath, dirs, files in os.walk(downloads_gisaid):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name, engine="xlrd")
            all_metadata_files.append(metadata)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name, states_ref) # Convert fasta file to dataframe
            all_fasta_files.append(fasta_file)

In [6]:
# Search for dates and states based on isolate

genotype_keys = {}
for og_key in original_fasta_dfs:
    key = og_key.split("/")[-1].split("_")[0]
    if key not in genotype_keys.keys():
        og_df = original_fasta_dfs[og_key]
        print(og_df)
        og_df["Unknown_States"] = og_df["Geo_Location"].apply(lambda x: 1 if x == "USA" else 0)
        og_df["Unknown_Dates"] = og_df["Date Collected"].apply(lambda x: 1 if dateutil.parser.parse(x, default=dateutil.parser.parse("2020-01-01")).month == dateutil.parser.parse("2025-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2020-01-01")).day == dateutil.parser.parse("2025-01-01").day else 0)
        og_df["Update_Needed"] = og_df["Unknown_States"] + og_df["Unknown_Dates"]
        # og_df["Identifier"] = ""
        og_df_update_needed = og_df[og_df["Update_Needed"] > 0]
        for isolate in og_df_update_needed["Isolate_Id"].values:
            # print(isolate)
            for new_df in all_fasta_files:
                if isolate in new_df["Isolate_Id"].values:
                    # print(isolate)
                    og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Geo_Location"] = new_df.loc[new_df[new_df["Isolate_Id"] == isolate].index[0], "Geo_Location"]
                    og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Date Collected"] = new_df.loc[new_df[new_df["Isolate_Id"] == isolate].index[0], "Date Collected"]
                    og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Identifier"] = new_df.loc[new_df[new_df["Isolate_Id"] == isolate].index[0], "Identifier"] # .split("|")[0]

        og_df_update_needed = og_df_update_needed.drop_duplicates(subset="Header")
        print(og_df_update_needed)

        # new_df = og_df.merge(og_df_update_needed, how="left")
        new_df = og_df.set_index('Header')
        new_df.update(og_df_update_needed.set_index('Header'))
        new_df = new_df.reset_index()
        # new_df = pd.concat([og_df_update_needed, og_df]).drop_duplicates(['Isolate_Id'], keep="last")
        new_df["Header"] = new_df["Identifier"] + "|" + new_df["Isolate_Name"] + "|" + new_df["Subtype"] + "|" + new_df["Geo_Location"] + "|" + new_df["Date Collected"] + "|" + new_df["Host_Type"] + "|" + new_df["Genotype"]
        print(new_df)
        # break 

        original_fasta_dfs[og_key] = new_df
        genotype_keys[key]= new_df["Header"]
    else:
        original_fasta_dfs[og_key]["Header"] = genotype_keys[key]

    

                                                Header  \
0    SRR33124713|A/CATTLE/USA/25-006783-001/2025|H5...   
1    SRR33124725|A/QUAIL/USA/25-006346-009/2025|H5N...   
2    SRR33124734|A/DUCK/USA/25-007078-002/2025|H5N1...   
3    SRR33124735|A/DUCK/USA/25-007078-001/2025|H5N1...   
4    SRR33124737|A/DUCK/USA/25-006674-002/2025|H5N1...   
..                                                 ...   
764  SRR33029749|A/gallus gallus/NY/25-010293-002-o...   
765  SRR33029739|A/gallus gallus/PA/25-010255-001-o...   
766  SRR33029738|A/gallus gallus/PA/25-010255-002-o...   
767  SRR33029737|A/gallus gallus/PA/25-010255-003-o...   
768  SRR33029746|A/anatidae/CA/24-035866-001-origin...   

                 Isolate_Id                                    Isolate_Name  \
0             25-006783-001                 A/CATTLE/USA/25-006783-001/2025   
1             25-006346-009                  A/QUAIL/USA/25-006346-009/2025   
2             25-007078-002                   A/DUCK/USA/25-007078

## Andersen

In [7]:
# Read metadata

os.chdir(home)
metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates

metadata_folder = home + "Andersen/avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv") # Everything else

print(len(metadata))

# Merge with metadata_normalized
metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

# # Find only >= last date using Release Date from metadata 
# metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")]
# # Find only <= update date using Release Date from metadata
# metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")]

print(len(metadata)) 
display(metadata)

9899
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
19252


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,ReleaseDate,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc
0,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,2023-06-30 00:46:13,2023-06-07 02:01:47,1,22-005893-001,SRP441379,H5N1,",",SRS17903639,False,NaN
1,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,2023-06-30 00:46:13,2023-06-07 02:01:47,1,22-005893-001,SRP441379,H5N1,"Swab, Tracheal",SRS17903639,False,NaN
2,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,2023-06-30 00:46:13,2023-06-07 02:01:22,1,AH0210318,SRP441379,H5N1,",/",SRS17903636,False,NaN
3,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,2023-06-30 00:46:13,2023-06-07 02:01:22,1,AH0210318,SRP441379,H5N1,"Swab Pool, Cloacal/Oropharyngeal",SRS17903636,False,NaN
4,SRR24839060,AMPLICON,201.66,21703662,PRJNA980729,SAMN35647621,Viral,10716272,United States Department of Agriculture,2022-02-17,...,2023-06-30 00:46:13,2023-06-07 02:01:28,1,22-005158-001,SRP441379,H5N1,NaN,SRS17903631,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19247,SRR34270260,WGS,148.04,146293566,PRJNA1207547,SAMN49684903,Viral,56219215,USDA-NVSL,2025,...,2025-06-27 13:16:36,2025-06-27 13:14:41,1,25-017747-001,SRP557452,NaN,CLOACAL/TRACHEAL SWAB POOL,SRS25589961,False,NaN
19248,SRR34270261,WGS,148.36,184527272,PRJNA1207547,SAMN49684902,Viral,70438887,USDA-NVSL,2025,...,2025-06-27 13:16:36,2025-06-27 13:14:41,1,25-017667-001,SRP557452,NaN,cloacal swab,SRS25589960,False,NaN
19249,SRR34270262,WGS,147.46,74065704,PRJNA1207547,SAMN49684901,Viral,29104205,USDA-NVSL,2025,...,2025-06-27 13:16:36,2025-06-27 13:14:37,1,25-017626-002,SRP557452,NaN,swab,SRS25589959,False,NaN
19250,SRR34270263,WGS,147.84,126106844,PRJNA1207547,SAMN49684892,Viral,45722422,USDA-NVSL,2024,...,2025-06-27 13:16:36,2025-06-27 13:14:41,1,24-028269-001,SRP557452,NaN,brain,SRS25589958,False,NaN


In [8]:
# Collapse dataset to only sequences in original dataset AND without dates OR states

genotype_keys = {}

for og_key in original_fasta_dfs:
    key = og_key.split("/")[-1].split("_")[0]
    print(key)
    if key not in genotype_keys.keys():
        og_df = original_fasta_dfs[og_key]
        print(og_df)
        # print(og_df)
        og_df["Unknown_States"] = og_df["Geo_Location"].apply(lambda x: 1 if x == "USA" else 0)
        og_df["Unknown_Dates"] = og_df["Date Collected"].apply(lambda x: 1 if dateutil.parser.parse(x, default=dateutil.parser.parse("2020-01-01")).month == dateutil.parser.parse("2025-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2020-01-01")).day == dateutil.parser.parse("2025-01-01").day else 0)
        og_df["Update_Needed"] = og_df["Unknown_States"] + og_df["Unknown_Dates"]
        # og_df["Identifier"] = ""
        og_df_update_needed = og_df[og_df["Update_Needed"] > 0]
        for isolate in og_df_update_needed["Isolate_Id"].values:
            # print(isolate)
            # for new_df in metadata:
            if isolate in metadata["isolate"].values:
                # print(isolate)
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Geo_Location"] = metadata.loc[metadata[metadata["isolate"] == isolate].index[0], "geo_loc_name"]
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Date Collected"] = metadata[metadata["isolate"] == isolate]["BioSample"].apply(lambda x: search_collection_date(x, metadata)).values[0]
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Identifier"] = metadata.loc[metadata[metadata["isolate"] == isolate].index[0], "Run"]

        print(og_df_update_needed)

        # new_df = og_df.merge(og_df_update_needed, how="left")
        # new_df = pd.concat([og_df_update_needed, og_df]).drop_duplicates('Isolate_Id', keep="first")
        new_df = og_df.set_index('Header')
        new_df.update(og_df_update_needed.set_index('Header'))
        new_df = new_df.reset_index()
        new_df["Header"] = new_df["Identifier"] + "|" + new_df["Isolate_Name"] + "|" + new_df["Subtype"] + "|" + new_df["Geo_Location"] + "|" + new_df["Date Collected"] + "|" + new_df["Host_Type"] + "|" + new_df["Genotype"]

        print(new_df)
        # break 

        original_fasta_dfs[og_key] = new_df
        genotype_keys[key]= new_df["Header"]
    else:
        original_fasta_dfs[og_key]["Header"] = genotype_keys[key]
    

D1
                                                Header  \
0    SRR33124713|A/CATTLE/USA/25-006783-001/2025|H5...   
1    SRR33124725|A/QUAIL/USA/25-006346-009/2025|H5N...   
2    SRR33124734|A/DUCK/USA/25-007078-002/2025|H5N1...   
3    SRR33124735|A/DUCK/USA/25-007078-001/2025|H5N1...   
4    SRR33124737|A/DUCK/USA/25-006674-002/2025|H5N1...   
..                                                 ...   
764  SRR33029749|A/gallus gallus/NY/25-010293-002-o...   
765  SRR33029739|A/gallus gallus/PA/25-010255-001-o...   
766  SRR33029738|A/gallus gallus/PA/25-010255-002-o...   
767  SRR33029737|A/gallus gallus/PA/25-010255-003-o...   
768  SRR33029746|A/anatidae/CA/24-035866-001-origin...   

                 Isolate_Id                                    Isolate_Name  \
0             25-006783-001                 A/CATTLE/USA/25-006783-001/2025   
1             25-006346-009                  A/QUAIL/USA/25-006346-009/2025   
2             25-007078-002                   A/DUCK/USA/25-007

## NCBI Virus

In [9]:
os.chdir(downloads_ncbi_virus)

# Read metadata
ncbi_metadata = pd.read_csv("sequences.csv")

# Integrate genotypes
# os.chdir(downloads_ncbi_virus + "11-01-2021--04-14-2025/")
# output = pd.read_csv("output.tsv", delimiter="\t")

# os.chdir(downloads_ncbi_virus + "04-14-2025--05-14-2025/")
# may_output = pd.read_csv("output.tsv", delimiter="\t")

print(ncbi_metadata)

        Accession GenBank_RefSeq SRA_Accession BioSample BioProject  \
0      PV576479.1        GenBank           NaN       NaN        NaN   
1      PV576480.1        GenBank           NaN       NaN        NaN   
2      PV576481.1        GenBank           NaN       NaN        NaN   
3      PV576482.1        GenBank           NaN       NaN        NaN   
4      PV576483.1        GenBank           NaN       NaN        NaN   
...           ...            ...           ...       ...        ...   
83492  OK205883.1        GenBank           NaN       NaN        NaN   
83493  OK205884.1        GenBank           NaN       NaN        NaN   
83494  OK205885.1        GenBank           NaN       NaN        NaN   
83495  OK205886.1        GenBank           NaN       NaN        NaN   
83496  OK205887.1        GenBank           NaN       NaN        NaN   

           Organism_Name                         Species Genotype    Isolate  \
0      Influenza A virus  Alphainfluenzavirus influenzae     H5N1  

In [10]:
# Search for dates and states based on isolate

genotype_keys = {}

for og_key in original_fasta_dfs:
    key = og_key.split("/")[-1].split("_")[0]
    print(key)
    if key not in genotype_keys.keys():
        og_df = original_fasta_dfs[og_key]
        print(og_df)
        # print(og_df)
        og_df["Unknown_States"] = og_df["Geo_Location"].apply(lambda x: 1 if x == "USA" else 0)
        og_df["Unknown_Dates"] = og_df["Date Collected"].apply(lambda x: 1 if dateutil.parser.parse(x, default=dateutil.parser.parse("2020-01-01")).month == dateutil.parser.parse("2025-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2020-01-01")).day == dateutil.parser.parse("2025-01-01").day else 0)

        # og_df["Unknown_Dates"] = og_df["Date Collected"].apply(dateutil.parser.parse).apply(lambda x: 1 if x == dateutil.parser.parse("2023-01-01") or x == dateutil.parser.parse("2024-01-01") or x == dateutil.parser.parse("2025-01-01") else 0)
        og_df["Update_Needed"] = og_df["Unknown_States"] + og_df["Unknown_Dates"]
        # og_df["Identifier"] = ""
        og_df_update_needed = og_df[og_df["Update_Needed"] > 0]
        # print(og_df_update_needed)
        for isolate in og_df_update_needed["Isolate_Id"].values:
            # print(isolate)
            # for new_df in ncbi_metadata:
                # print(new_df)
            if isolate in ncbi_metadata["Isolate"].values:
                print(isolate)
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Geo_Location"] = ncbi_metadata.loc[ncbi_metadata[ncbi_metadata["Isolate"] == isolate].index[0], "Geo_Location"] # .replace(": ", "-")
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Date Collected"] = ncbi_metadata.loc[ncbi_metadata[ncbi_metadata["Isolate"] == isolate].index[0], "Collection_Date"]
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Identifier"] = ncbi_metadata.loc[ncbi_metadata[ncbi_metadata["Isolate"] == isolate].index[0], "SRA_Accession"]

        # new_df = og_df.merge(og_df_update_needed, how="left")
        # new_df = pd.concat([og_df_update_needed, og_df]).drop_duplicates('Isolate_Id', keep="first")

        new_df = og_df.set_index('Header')
        new_df.update(og_df_update_needed.set_index('Header'))
        new_df = new_df.reset_index()
        new_df["Header"] = new_df["Identifier"] + "|" + new_df["Isolate_Name"] + "|" + new_df["Subtype"] + "|" + new_df["Geo_Location"] + "|" + new_df["Date Collected"] + "|" + new_df["Host_Type"] + "|" + new_df["Genotype"]

        print(new_df)
        # break 

        original_fasta_dfs[og_key] = new_df
        genotype_keys[key]= new_df["Header"]
    else:
        original_fasta_dfs[og_key]["Header"] = genotype_keys[key]
    

D1
                                                Header  \
0    SRR33124713|A/CATTLE/USA/25-006783-001/2025|H5...   
1    SRR33124725|A/QUAIL/USA/25-006346-009/2025|H5N...   
2    SRR33124734|A/DUCK/USA/25-007078-002/2025|H5N1...   
3    SRR33124735|A/DUCK/USA/25-007078-001/2025|H5N1...   
4    SRR33124737|A/DUCK/USA/25-006674-002/2025|H5N1...   
..                                                 ...   
764  SRR33029749|A/gallus gallus/NY/25-010293-002-o...   
765  SRR33029739|A/gallus gallus/PA/25-010255-001-o...   
766  SRR33029738|A/gallus gallus/PA/25-010255-002-o...   
767  SRR33029737|A/gallus gallus/PA/25-010255-003-o...   
768  SRR33029746|A/anatidae/CA/24-035866-001-origin...   

                 Isolate_Id                                    Isolate_Name  \
0             25-006783-001                 A/CATTLE/USA/25-006783-001/2025   
1             25-006346-009                  A/QUAIL/USA/25-006346-009/2025   
2             25-007078-002                   A/DUCK/USA/25-007

# Put it all together

In [11]:
def df_to_fasta(fasta, file_name, output_path):

    output_file = open(output_path + file_name, "w")

    for index, row in fasta.iterrows():
        name = fasta.loc[index, "full_header"]
        print(name)
        sequence = fasta.loc[index, "sequence"]
    # First is header, second is sequence
        output_file.write(name + "\n")
        output_file.write(sequence + "\n")
    output_file.close()

for key in original_fasta_dfs:
    df = original_fasta_dfs[key]
    # df["Identifier"] = df["Identifier"].apply(lambda x: "unknown" if x != x else x) # Put "unknown" if NaN
    # df = df[df["Genotype"] == genotype] # Make sure we only have the genotype we want
    df["Header"].fillna("unknown", inplace=True)
    df["full_header"] = ">" + df["Header"]
    df["sequence"] = df["Sequence"]

    df_to_fasta(df, key.split("/")[-1][:-6] + "_updated_" + update_date + ".fasta", complete)

>SRR33124713|A/CATTLE/USA/25-006783-001/2025|H5N1|USA|2025|cattle|D1.1
>SRR33124725|A/QUAIL/USA/25-006346-009/2025|H5N1|USA|2025|avian|D1.1
>SRR33124734|A/DUCK/USA/25-007078-002/2025|H5N1|USA|2025-02-24|avian|D1.1
>SRR33124735|A/DUCK/USA/25-007078-001/2025|H5N1|USA|2025-02-24|avian|D1.1
>SRR33124737|A/DUCK/USA/25-006674-002/2025|H5N1|USA|2025-02-24|avian|D1.1
>SRR33124748|A/CHICKEN/USA/25-007056-001/2025|H5N1|USA|2025-02-17|avian|D1.1
>SRR33124769|A/CHICKEN/USA/25-006346-008/2025|H5N1|USA|2025-02-20|avian|D1.1
>SRR33124770|A/CHICKEN/USA/25-006346-007/2025|H5N1|USA|2025-02-20|avian|D1.1
>SRR33124773|A/CHICKEN/USA/25-006346-004/2025|H5N1|USA|2025-02-20|avian|D1.1
>SRR33125016|A/DUCK/USA/25-004480-001/2025|H5N1|USA|2025|avian|D1.1
>SRR33125020|A/BALD EAGLE/USA/25-005869-001/2025|H5N1|USA|2025|avian|D1.1
>SRR33125021|A/CAT/USA/25-006047-001/2025|H5N1|USA|2025|feline|D1.1
>SRR33125028|A/CANADA GOOSE/USA/25-004415-006/2025|H5N1|USA|2025|avian|D1.1
>SRR33125032|A/CANADA GOOSE/USA/25-004376-00

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_23672\3742918428.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Header"].fillna("unknown", inplace=True)
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_23672\3742918428.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

>EPI_ISL_19870609|A/guineafowl/USA/010935-003/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19851153|A/peregrine/USA/038450-001/2024|H5N1|USA|2024-01-01|avian|D1.1
>EPI_ISL_19851244|A/duck/USA/011483-005/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19851235|A/duck/USA/011552-007/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19851262|A/chicken/USA/012098-004/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19851253|A/chicken/USA/012099-002/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19851206|A/guineafowl/USA/012039-002/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19851204|A/turkey/USA/011106-004/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19851205|A/turkey/USA/011106-001/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19851202|A/chicken/USA/011365-003/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19851203|A/turkey/USA/011532-001/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19851201|A/chicken/USA/011483-002/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19851226|A/duck/USA/011552-008/2025|H5N1|USA

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_23672\3742918428.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Header"].fillna("unknown", inplace=True)
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_23672\3742918428.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

>EPI_ISL_19851288|A/chicken/USA/011365-002/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19851289|A/skunk/USA/011279-001/2025|H5N1|USA|2025-01-01|other_mammal|D1.1
>EPI_ISL_19851286|A/chicken/USA/011237-001/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19851287|A/turkey/USA/011106-003/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19859579|A/red_fox/USA/012694-001/2025|H5N1|USA|2025-01-01|other_mammal|D1.1
>EPI_ISL_19859577|A/rock_goose/USA/012362-001/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19859576|A/red-tailed_hawk/USA/011782-001/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19859631|A/great_horned_owl/USA/012288-002/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19859630|A/great_horned_owl/USA/007118-029-R2/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19859629|A/great_horned_owl/USA/012397-001/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19859623|A/red-tailed_hawk/USA/012695-001/2025|H5N1|USA|2025-01-01|avian|D1.1
>EPI_ISL_19859621|A/red-tailed_hawk/USA/012741-002/2025|H5N1|USA|2025-0

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\2\ipykernel_23672\3742918428.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Header"].fillna("unknown", inplace=True)
